# TraceIQ — PDF Question Answering with Hybrid RAG

This notebook builds a system that reads a PDF and answers questions about it.

Instead of using a simple search, it combines several techniques to make retrieval much better:

- **PDF parsing** — extracts text page by page using PyPDF
- **Semantic chunking** — splits text where meaning changes, not at fixed word counts
- **BGE embeddings** — converts text into vectors that capture meaning
- **Qdrant vector database** — stores and searches those vectors
- **BM25 keyword search** — finds exact keyword matches that semantic search can miss
- **Reciprocal Rank Fusion (RRF)** — merges the two ranked lists into one better list
- **Cross-encoder reranking** — re-scores each chunk by reading the query and chunk together
- **Dynamic context budgeting** — fills a token budget with the best chunks, nothing more
- **Source tracking** — shows which page each answer came from

**How to run:** upload a PDF or paste text into the Gradio interface at the bottom, then ask a question.

**API key needed:** `GROQ_API_KEY` in Colab Secrets (left sidebar → key icon).

**Folder:** `01_TraceIQ_PDF_RAG/`


In [ ]:
!pip install -q pypdf rank-bm25 qdrant-client gradio groq google-generativeai
print("Done.")

In [ ]:
# Ensure the notebook uses the expected Gemini SDK version
# import importlib, google.generativeai
# importlib.reload(google.generativeai)
# import google.generativeai as genai
# print("genai ready.")


In [ ]:
import os, uuid, time
import numpy as np
from typing import List, Dict, Tuple, Optional, Any
from pypdf import PdfReader
from rank_bm25 import BM25Okapi
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
import gradio as gr

CONFIG = {
    # Gemini embedding-004 returns 768-dimensional embeddings
    "vector_dim"       : 768,
    "gemini_model"     : "gemini-2.0-flash",
    "embed_model"      : "models/text-embedding-004",
    "qdrant_collection": "traceiq",
    "rrf_k"            : 60,
    "top_k_retrieve"   : 10,
    "top_k_final"      : 3,
    "chunk_size"       : 300,
    "chunk_overlap"    : 40,
}
print("Config ready.")


Config ready.


In [ ]:
from google.colab import userdata
from groq import Groq

try:
    raw = userdata.get("GROQ_API_KEY")
    GROQ_KEY = str(raw).strip() if not isinstance(raw, dict) else (
        raw.get("data", {}).get("payload") or raw.get("payload") or ""
    )
except Exception:
    GROQ_KEY = ""

if not GROQ_KEY:
    GROQ_KEY = input("Paste your Groq API key: ").strip()

groq_client = Groq(api_key=GROQ_KEY)

def llm(prompt: str) -> str:
    resp = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        max_tokens=1024,
    )
    return resp.choices[0].message.content.strip()

# Verify that the embedding model is working correctly
print("Groq test:", llm("Say: working"))

Groq test: Working.


In [ ]:
!pip install -q chromadb

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.6 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.19.5, but you have protobuf 6.33.6 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-exporter-otlp-proto-common==1.38.0, but you have opentelemetry-exporter-otlp-proto-common 1.42.1 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-proto==1.38.0, but you have opentelemetry-proto 1.42.1 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-sdk~=1.38.0, but you have opentelemetry-sdk 1.42.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.42.1 which is incompatible.
google-adk 1.29.0 requires 

In [ ]:
#Text to vector
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("BAAI/bge-small-en-v1.5")

def embed_texts(texts):
    return model.encode(texts, normalize_embeddings=True)

def embed_query(text):
    return model.encode(text, normalize_embeddings=True)

CONFIG["vector_dim"] = 384

v = embed_query("what is self attention")
print(f"Embedding works. Vector dim: {len(v)}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding works. Vector dim: 384


## PDF Processing Pipeline

The document goes through two steps before it can be searched:

1. **Text extraction** — PyPDF reads each page and saves its text along with the page number.
2. **Semantic chunking** — instead of splitting at a fixed number of words, the text is split where the topic changes. This keeps related sentences together in the same chunk.

Each chunk keeps its page number so the final answer can show exactly where the information came from.


In [ ]:
pip install langchain-experimental langchain-community

In [ ]:
# PDF processing utilities
# parse_pdf extracts text and page metadata
# chunk_text splits content into overlapping chunks for retrieval

import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

def estimate_tokens(text: str) -> int:
    return len(enc.encode(text))

from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)

semantic_splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

def chunk_text(text: str):
    chunks = semantic_splitter.split_text(text)
    return [c.strip() for c in chunks if c.strip()]


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# Generate a document-level summary.
# This summary is stored separately and attached during retrieval,
# not embedded into every chunk.

def extract_global_pdf_context(text: str) -> str:
    prompt = (
        f"Analyze the following document text and write a highly condensed "
        f"two-sentence summary defining the exact system, hardware, or software "
        f"described, its name, and its main goal. "
        f"Do not write any introductory words or filler:\n\n{text[:8000]}"
    )

    try:
        return llm(prompt)
    except Exception as e:
        print(f"⚠️ Context extraction failed: {e}")
        return "Technical documentation manual."


def build_index(pdf_path, paste_text):

    all_chunks = []

    # PDF INGESTION
    if pdf_path and os.path.exists(pdf_path):

        pages = parse_pdf(pdf_path)
        fname = os.path.basename(pdf_path)

        print(f"PDF: {fname} — {len(pages)} pages")

        # Generate document-level summary
        full_text_sample = " ".join(
            [p["text"] for p in pages[:5]]
        )

        global_context = extract_global_pdf_context(
            full_text_sample
        )

        print(
            f"✨ Global PDF Context Extracted:\n"
            f"{global_context}\n"
        )

        before = len(all_chunks)

        for p in pages:

            chunks = chunk_text(p["text"])

            for i, ct in enumerate(chunks, 1):

                all_chunks.append({

                    "id": str(uuid.uuid4()),

                    "source": f"{fname} p.{p['page']}",

                    "name": f"p{p['page']}-c{i}",

                    "type": "pdf",

                    # Actual text chunk used for embeddings
                    "content": ct,

                    # Parent document summary
                    "parent_context": global_context,

                    "tokens": estimate_tokens(ct),

                })

        print(
            f"  {len(all_chunks) - before} chunks from PDF"
        )


    # PASTED TEXT INGESTION
    if paste_text and paste_text.strip():

        global_context = extract_global_pdf_context(
            paste_text
        )

        print(
            f"✨ Global Text Context Extracted:\n"
            f"{global_context}\n"
        )

        before = len(all_chunks)

        for i, ct in enumerate(
            chunk_text(paste_text), 1
        ):

            all_chunks.append({

                "id": str(uuid.uuid4()),

                "source": "pasted-text",

                "name": f"s{i}",

                "type": "text",

                # Actual text chunk used for embeddings
                "content": ct,

                # Parent document summary
                "parent_context": global_context,

                "tokens": estimate_tokens(ct),

            })

        print(
            f"  {len(all_chunks) - before} chunks from pasted text"
        )

    if not all_chunks:
        raise ValueError(
            "Nothing to index. Upload a PDF or paste text."
        )


    # EMBEDDINGS
    print(f"Embedding {len(all_chunks)} chunks...")

    texts = [c["content"] for c in all_chunks]

    vecs = embed_texts(texts)

    # QDRANT
    qc = QdrantClient(location=":memory:")

    qc.create_collection(
        collection_name=CONFIG["qdrant_collection"],
        vectors_config=VectorParams(
            size=CONFIG["vector_dim"],
            distance=Distance.COSINE,
        ),
    )

    qc.upsert(
        collection_name=CONFIG["qdrant_collection"],
        points=[
            PointStruct(
                id=i,
                vector=vecs[i].tolist(),
                payload=c,
            )
            for i, c in enumerate(all_chunks)
        ],
    )

    for i, c in enumerate(all_chunks):
        c["idx"] = i

    print("  Qdrant vector index ready.")


    # BM25
    bm25 = BM25Okapi(
        [c["content"].lower().split() for c in all_chunks]
    )

    print("  BM25 index ready.")

    total_tok = sum(c["tokens"] for c in all_chunks)

    print(
        f"Done. {len(all_chunks)} chunks, "
        f"~{total_tok:,} tokens indexed."
    )

    return all_chunks, qc, bm25


print("build_index() ready.")


build_index() ready.


In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

print("Cross Encoder loaded.")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Cross Encoder loaded.


In [ ]:
def retrieve(query, chunks, qc, bm25):

    # Stage 1: Dense Retrieval
    q_vec = embed_query(query).tolist()

    dense_res = qc.query_points(
        collection_name=CONFIG["qdrant_collection"],
        query=q_vec,
        limit=CONFIG["top_k_retrieve"],
    ).points

    dense_ranks = {
        int(h.id): rank
        for rank, h in enumerate(dense_res)
    }

    # Stage 2: BM25 Retrieval
    bm25_scores = bm25.get_scores(
        query.lower().split()
    )

    top_bm25_idx = np.argsort(
        bm25_scores
    )[::-1][:CONFIG["top_k_retrieve"]]

    bm25_ranks = {
        int(i): rank
        for rank, i in enumerate(top_bm25_idx)
    }

    # Stage 3: RRF Fusion
    k = CONFIG["rrf_k"]

    all_idxs = (
        set(dense_ranks)
        | set(bm25_ranks)
    )

    rrf_scores = {
        idx: (
            1 / (k + dense_ranks.get(idx, CONFIG["top_k_retrieve"]))
            + 1 / (k + bm25_ranks.get(idx, CONFIG["top_k_retrieve"]))
        )
        for idx in all_idxs
    }

    top_idxs = {
        idx
        for idx, _ in sorted(
            rrf_scores.items(),
            key=lambda x: x[1],
            reverse=True
        )[:CONFIG["top_k_retrieve"]]
    }

    candidates = [
        c
        for c in chunks
        if c["idx"] in top_idxs
    ]

    if not candidates:
        return []

    # Stage 4: Cross Encoder Reranking

    pairs = [
        (query, c["content"])
        for c in candidates
    ]

    scores = reranker.predict(pairs)

    for c, score in zip(candidates, scores):
        c["score"] = float(score)

    candidates.sort(
        key=lambda c: c["score"],
        reverse=True
    )

    return candidates[
        : CONFIG["top_k_final"]
    ]

In [ ]:
def ask(query, chunks):

    context_budget = 3000

    selected_chunks = []
    current_tokens = 0

    for c in chunks:

        chunk_tokens = c["tokens"]

        if current_tokens + chunk_tokens > context_budget:
            break

        selected_chunks.append(c)
        current_tokens += chunk_tokens

    context = "\n\n".join(
        f"[{c['source']}:{c['name']}]\n{c['content']}"
        for c in selected_chunks
    )

    prompt = f"""
Answer the question using ONLY the provided context.

If the answer is not present, say:
"The document does not contain this information."

QUESTION:
{query}

CONTEXT:
{context}

ANSWER:
"""

    answer = llm(prompt)

    return {
        "answer": answer,
        "sources": [
            {
                "source": c["source"],
                "chunk": c["name"]
            }
            for c in selected_chunks
        ],
        "in_tokens": estimate_tokens(prompt),
        "out_tokens": estimate_tokens(answer)
    }

In [ ]:
STATE = {"chunks": [], "qc": None, "bm25": None, "ready": False}

def ui_build(pdf_file, paste_text):
    has_pdf  = pdf_file is not None
    has_text = bool(paste_text and paste_text.strip())
    if not has_pdf and not has_text:
        return "Please upload a PDF or paste text first."
    try:
        chunks, qc, bm25 = build_index(
            pdf_file.name if has_pdf else None,
            paste_text    if has_text else None,
        )
        STATE.update(chunks=chunks, qc=qc, bm25=bm25, ready=True)
        pdf_n  = sum(1 for c in chunks if c["type"] == "pdf")
        text_n = sum(1 for c in chunks if c["type"] == "text")
        tok    = sum(c["tokens"] for c in chunks)
        return (
            f"Ready.\n"
            f"Total chunks : {len(chunks)}\n"
            f"PDF chunks   : {pdf_n}\n"
            f"Text chunks  : {text_n}\n"
            f"Total tokens : ~{tok:,}\n\n"
            f"Ask your question on the right."
        )
    except Exception as e:
        import traceback
        STATE["ready"] = False
        return f"Build failed:\n{traceback.format_exc()}"

def ui_query(question):

    if not STATE["ready"]:
        return "Build the knowledge base first.", ""

    if not question or not question.strip():
        return "Please type a question.", ""

    try:
        t0 = time.time()

        top = retrieve(
            question,
            STATE["chunks"],
            STATE["qc"],
            STATE["bm25"]
        )

        if not top:
            return "No relevant content found. Try rephrasing.", ""

        res = ask(question, top)

        elapsed = time.time() - t0

        # Context reduction metrics
        total_kb_tokens = sum(
            c["tokens"]
            for c in STATE["chunks"]
        )

        saved_tokens = max(
            0,
            total_kb_tokens - res["in_tokens"]
        )

        saved_pct = (
            saved_tokens / total_kb_tokens
        ) * 100 if total_kb_tokens > 0 else 0

        reduction_blocks = min(
            10,
            max(0, round(saved_pct / 10))
        )

        reduction_bar = (
            "█" * reduction_blocks
            + "░" * (10 - reduction_blocks)
        )

        # Sources
        sources_md = "\n".join(
            f"- `{s['source']}` ({s['chunk']})"
            for s in res["sources"]
        )

        # Final answer card
        answer_md = (
            f"## Answer\n\n"
            f"{res['answer']}\n\n"
            f"### Sources\n"
            f"{sources_md}\n\n"
            f"---\n"
            f"⚡ **Token Compression** "
            f"{reduction_bar} "
            f"{saved_pct:.0f}% context compressed\n\n"
            f"Context sent to LLM: "
            f"`{res['in_tokens']:,}` tokens "
            f"(vs naive `{total_kb_tokens:,}` full-text tokens)\n\n"
            f"Latency: `{elapsed:.2f}s`\n\n"
            f"Generation: `{res['out_tokens']:,}` tokens"
        )

        # Retrieval trace
        trace_rows = []

        for i, c in enumerate(top, 1):

            score = c.get("score", 0)

            # Cross encoder scores are not %
            score_percent = max(
                0,
                min(100, score * 10)
            )

            score_blocks = min(
                10,
                max(0, round(score_percent / 10))
            )

            score_bar = (
                "█" * score_blocks
                + "░" * (10 - score_blocks)
            )

            snippet = c.get(
                "original_content",
                c["content"]
            )[:400]

            trace_rows.append(
                f"### {i}. {c['source']} / {c['name']}\n\n"
                f"Relevance: "
                f"{score_bar} "
                f"{score_percent:.1f}%\n\n"
                f"```text\n"
                f"{snippet}\n"
                f"```"
            )

        trace = (
            "## Retrieved Context Chunks\n\n"
            + "\n\n".join(trace_rows)
        )

        return answer_md, trace

    except Exception as e:
        import traceback

        return (
            f"Query failed:\n{traceback.format_exc()}",
            ""
        )

css = """
body, .gradio-container { background:#0f172a; color:#e2e8f0; font-family:monospace; }
textarea, input { background:#1e293b !important; color:#e2e8f0 !important; border:1px solid #334155 !important; }
label { color:#94a3b8 !important; }
"""

with gr.Blocks(title="TraceIQ", css=css, theme=gr.themes.Base()) as app:
    gr.Markdown("# TraceIQ — PDF Retrieval and Question Answering")
    with gr.Row():
        with gr.Column(scale=4):
            gr.Markdown("### Step 1 — Data")
            pdf_in     = gr.File(label="PDF file", file_types=[".pdf"])
            paste_in   = gr.Textbox(label="Or paste text", lines=4, placeholder="Paste any text...")
            build_btn  = gr.Button("Build knowledge base", variant="primary")
            status_out = gr.Textbox(label="Log", lines=10, interactive=False)
        with gr.Column(scale=6):
            gr.Markdown("### Step 2 — Ask")
            q_in    = gr.Textbox(label="Ask Question", lines=3, placeholder="e.g. What is self-attention?")
            ask_btn = gr.Button("Get answer", variant="primary")
            ans_out = gr.Markdown("Answer will appear here.")
            with gr.Accordion("Chunks used", open=False):
                trace_out = gr.Markdown("Retrieval detail appears here.")

    build_btn.click(ui_build, [pdf_in, paste_in], [status_out])
    ask_btn.click(ui_query, [q_in], [ans_out, trace_out])

print("Launching...")
app.launch(share=True, debug=False)


/tmp/ipykernel_30873/472941402.py:162: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="TraceIQ", css=css, theme=gr.themes.Base()) as app:
/tmp/ipykernel_30873/472941402.py:162: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(title="TraceIQ", css=css, theme=gr.themes.Base()) as app:


Launching...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://da7748c06396112550.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# Run a local retrieval test without the Gradio interface
# Useful for validating indexing and retrieval during dev

if STATE["ready"]:
    q   = "What is the key component of the Transformer architecture?"
    print("Query:", q)
    print("-" * 60)
    t0 = time.time()
    top = retrieve(q, STATE["chunks"], STATE["qc"], STATE["bm25"])
    if not top:
        print("No chunks retrieved. Something is wrong with the index.")
    else:
        res = ask(q, top)
        elapsed = time.time() - t0
        print(res["answer"])
        print()
        print(f"Latency: {elapsed:.2f}s | "f"Context sent: {res['in_tokens']} tokens | "f"Generated: {res['out_tokens']} tokens")
        print("-" * 60)
        for i, c in enumerate(top, 1):
            score = c.get("score", 0)
            score_blocks = min(10, max(0, round(score * 10)))
            score_bar = "█" * score_blocks + "░" * (10 - score_blocks)
            print(f"  {i}. {c['source']} / {c['name']} — Relevance: {score_bar} {score*100:.1f}% Match")
else:
     # Build a small sample index for testing when the UI has not been used.
    print("Building sample index...")
    chunks, qc, bm25 = build_index(None, "The self attention mechanism is the key component of the transformer architecture.")
    STATE.update(chunks=chunks, qc=qc, bm25=bm25, ready=True)

    q   = "What is the key component of the Transformer architecture?"
    top = retrieve(q, STATE["chunks"], STATE["qc"], STATE["bm25"])
    res = ask(q, top)
    print("Q:", q)
    print()
    print(res["answer"])


Building sample index...
✨ Global Text Context Extracted:
The system described is the Transformer architecture, a neural network model. Its main goal is to handle sequential data using the self-attention mechanism as its key component.

  1 chunks from pasted text
Embedding 1 chunks...
  Qdrant vector index ready.
  BM25 index ready.
Done. 1 chunks, ~13 tokens indexed.
Q: What is the key component of the Transformer architecture?

The self-attention mechanism.
